In [1]:
# 向量点积

import torch

a = torch.randn(3)  # [3]
b = torch.randn(3)  # [3]

# "i,i->" : i重复→求和，输出无维度→标量
c = torch.einsum("i,i->", a, b)
# 等价: torch.dot(a, b) 或 (a * b).sum()

print(c.shape)  # torch.Size([])
print(c)

torch.Size([])
tensor(1.4820)


In [ ]:
# 矩阵乘法
A = torch.randn(2, 3)  # [2, 3]
B = torch.randn(3, 4)  # [3, 4]

# "ij,jk->ik" : j重复→求和，保留i和k
C = torch.einsum("ij,jk->ik", A, B)
# 等价: A @ B 或 torch.matmul(A, B)

print(C.shape)  # [2, 4]

In [2]:
# 批量矩阵乘法

A = torch.randn(10, 3, 4)  # [batch, m, n]
B = torch.randn(10, 4, 5)  # [batch, n, p]

# 方法1: 显式指定batch维度
C1 = torch.einsum("bij,bjk->bik", A, B)  # [10, 3, 5]

# 方法2: 使用省略号（推荐）
C2 = torch.einsum("...ij,...jk->...ik", A, B)

print(C1.shape, C2.shape)  # 都是 [10, 3, 5]

torch.Size([10, 3, 5]) torch.Size([10, 3, 5])


| 操作         | einsum 表达式       | 等价函数             |
| ---------- | ---------------- | ---------------- |
| **向量点积**   | `"i,i->"`        | `torch.dot`      |
| **向量外积**   | `"i,j->ij"`      | `torch.outer`    |
| **矩阵乘法**   | `"ij,jk->ik"`    | `torch.mm` / `@` |
| **批量矩阵乘**  | `"bij,bjk->bik"` | `torch.bmm`      |
| **矩阵转置**   | `"ij->ji"`       | `tensor.T`       |
| **矩阵对角线**  | `"ii->i"`        | `torch.diag`     |
| **矩阵迹**    | `"ii->"`         | `torch.trace`    |
| **批量对角线**  | `"...ii->...i"`  | —                |
| **元素乘法求和** | `"ij,ij->"`      | `(A*B).sum()`    |
| **加权求和**   | `"bij,bj->bi"`   | —                |


In [3]:
# 三个矩阵的复杂操作
A = torch.randn(3, 4)
B = torch.randn(4, 5)
C = torch.randn(5, 6)

# A @ B @ C 的分解
D = torch.einsum("ij,jk,kl->il", A, B, C)
print(D.shape)  # [3, 6]

# 或者保留中间维度
E = torch.einsum("ij,jk,kl->ijkl", A, B, C)
print(E.shape)  # [3, 4, 5, 6]

torch.Size([3, 6])
torch.Size([3, 4, 5, 6])


In [4]:
x = torch.randn(2, 3, 4, 5)  # [batch, seq, head, dim]

# 重排为 [batch, head, seq, dim]
y = torch.einsum("bshd->bhsd", x)
# 等价: x.permute(0, 2, 1, 3)

print(y.shape)  # [2, 4, 3, 5]

torch.Size([2, 4, 3, 5])


In [ ]:
def attention(Q, K, V):
    """
    Q, K, V: [batch, heads, seq, dim]
    """
    # 1. Q @ K^T
    scores = torch.einsum("bhqd,bhkd->bhqk", Q, K)
    
    # 2. Softmax
    attn = torch.softmax(scores / (Q.size(-1) ** 0.5), dim=-1)
    
    # 3. Attn @ V
    output = torch.einsum("bhqk,bhkd->bhqd", attn, V)
    
    return output, attn

# 测试
B, H, N, D = 2, 8, 100, 64
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

out, attn = attention(Q, K, V)
print(out.shape)   # [2, 8, 100, 64]
print(attn.shape)  # [2, 8, 100, 100]

In [ ]:
def moe_forward(x, gates, expert_weights):
    """
    x: [batch, seq, dim] = [B, S, D]
    gates: [batch, seq, num_experts] = [B, S, E]  # 门控分数
    expert_weights: [num_experts, dim, out_dim] = [E, D, O]
    """
    # 每个token过所有专家，然后按gate加权
    # x @ expert_weights^T: [B,S,D] @ [E,D,O] -> [B,S,E,O]
    expert_outputs = torch.einsum("bsd,edo->bseo", x, expert_weights)
    
    # gates加权: [B,S,E] @ [B,S,E,O] -> [B,S,O]
    # 注意：这里用元素乘+求和，不是矩阵乘
    output = torch.einsum("bse,bseo->bso", gates, expert_outputs)
    
    return output

# 测试
B, S, D, E, O = 2, 10, 64, 4, 128
x = torch.randn(B, S, D)
gates = torch.softmax(torch.randn(B, S, E), dim=-1)  # 归一化
expert_weights = torch.randn(E, D, O)

out = moe_forward(x, gates, expert_weights)
print(out.shape)  # [2, 10, 128]

In [5]:
# 调试：维度检查

def safe_einsum(equation, *tensors):
    """安全的einsum，打印维度信息"""
    print(f"Equation: {equation}")
    for i, t in enumerate(tensors):
        print(f"  Tensor {i}: shape={t.shape}")
    
    try:
        result = torch.einsum(equation, *tensors)
        print(f"  Output: shape={result.shape}")
        return result
    except RuntimeError as e:
        print(f"  Error: {e}")
        raise

# 使用
A = torch.randn(3, 4)
B = torch.randn(4, 5)
safe_einsum("ij,jk->ik", A, B)

Equation: ij,jk->ik
  Tensor 0: shape=torch.Size([3, 4])
  Tensor 1: shape=torch.Size([4, 5])
  Output: shape=torch.Size([3, 5])


tensor([[ 0.8294, -0.5614, -1.2259, -1.5681, -0.9829],
        [ 0.4267, -1.6226, -0.8456, -1.8639, -0.2647],
        [-1.5721, -0.1295,  0.4425,  1.9213, -0.4320]])

# 性能对比
- 结论：einsum 会略微慢于专用函数（如matmul），但差距很小，换来的是代码可读性和灵活性。
- 极端性能场景不如专用函数

In [ ]:
import torch
import time

A = torch.randn(100, 200, 300)
B = torch.randn(100, 300, 400)

# 方法1: einsum
start = time.time()
for _ in range(100):
    C1 = torch.einsum("bij,bjk->bik", A, B)
t1 = time.time() - start

# 方法2: matmul
start = time.time()
for _ in range(100):
    C2 = torch.matmul(A, B)
t2 = time.time() - start

print(f"einsum: {t1:.4f}s")
print(f"matmul: {t2:.4f}s")
print(f"结果相同: {torch.allclose(C1, C2)}")

In [ ]:
einsum 的核心优势：
┌─────────────────────────────────────────┐
│  ✅ 一行代码替代多种操作（乘、转置、求和、对角等）│
│  ✅ 显式维度命名，代码自文档化               │
│  ✅ 自动处理广播，减少reshape               │
│  ✅ 与NumPy/TensorFlow/JAX语法通用          │
│  ⚠️  复杂表达式可读性下降，需注释            │
│  ⚠️  极端性能场景不如专用函数               │
└─────────────────────────────────────────┘